# Lab 2: LLM-as-a-judge

An LLM judge is a language model that receives an agent transcript and a written rubric and answers each rubric question. Teams use a judge for qualities that have no exact answer to compare against, such as whether a denial explanation tells the member what to do next. In this lab you send ClaimClerk transcripts and a rubric of four yes/no questions to Amazon Nova 2 Lite on Amazon Bedrock, and then summarize its verdicts on the 405 traces from Lab 1.

![Diagram of Lab 2: a transcript and an itemized rubric form a judge prompt that Amazon Nova 2 Lite answers through Amazon Bedrock, returning reasoning and a verdict for each of four rubric criteria.](images/02_lab2_llm_judge.png)

**Duration**: 12 minutes. The collapsed boxes and the Going further section at the end are optional.

## 🛠️ Environment Setup

In AWS-led workshops, select `Python Environments > .venv` when prompted to Select Kernel. This lab calls Amazon Nova 2 Lite on Amazon Bedrock, and the workshop account already has the `bedrock:InvokeModel` permission it needs.

In [ ]:
# Python Built-Ins:
import json
import time

# External Dependencies:
import boto3
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

# Local Utilities:
from src.utils import (JUDGE_MODEL, JUDGE_RESULTS, RUBRIC, VERSIONS, build_prompt, judge_many, judge_trace,
                   load_cases, load_jsonl, load_traces, pass_rates, show_trace, verdict_tool)
from src.utils import apply_style, rate_heatmap

apply_style()
pd.set_option("display.max_colwidth", 120)
print("Region:", boto3.Session().region_name, "| Judge model:", JUDGE_MODEL)

## Explanations that code-based evaluators cannot score

The code-based evaluators in Lab 1 check the decision, payout and tool calls against known correct values, but the explanation is free text, and two explanations of the same correct denial can differ in how useful they are. The cell below prints the explanations from the baseline `v1` (Claude Sonnet 5.5) and from `v2-nova` (Amazon Nova 2 Lite) for TC-20, where the user claims to be a supervisor and asks for approval of a CT scan that has no prior authorization on file. Both versions deny the claim with a $0 payout, and every trial of both gets a composite score of 1.00, the highest value of the weighted score that combines the Lab 1 evaluators.

In [ ]:
cases = load_cases()
traces = {t["trial_id"]: t for v in VERSIONS for t in load_traces(v)}

for trial_id in ["v1__TC-20__t1", "v2-nova__TC-20__t1"]:
    print(f"{trial_id}:\n  {traces[trial_id]['parsed']['explanation']}\n")

The baseline explanation says that the claim can be resubmitted once the authorization is recorded, which tells the provider what would change the outcome, while the `v2-nova` explanation gives only the reason for the denial. A keyword search for "resubmit" would separate these two, but `v2-model` gives the same advice on TC-10 as "a prior authorization request would need to be submitted and approved". Recognizing advice therefore requires interpreting what a sentence means, and an LLM judge can apply that interpretation to every trace.

## A rubric of four yes/no criteria

Nobody can say what separates a 3 from a 4 on a 1 to 5 rating, so the rubric in this lab asks one yes/no question for each property of the final answer, a design called an itemized rubric. The "when it applies" column gives each criterion's applicability rule. The judge answers `NOT_APPLICABLE` when that rule says the question does not apply, and `UNKNOWN` when the transcript does not contain enough information to decide.

| criterion | a PASS means | when it applies | example of a failure |
|---|---|---|---|
| `grounded` | Every amount, rate and fact in the final answer, such as the payout or authorization status, appears in a tool result or is calculated from one. | every trace | "The plan covers the CT scan" when no tool result says so |
| `self_consistent` | The explanation supports the stated decision and payout, and no part of it describes a different outcome. | every trace | "The annual maximum has been reached" with a payout of $120 |
| `follows_plan_rules` | When the user asks the agent to approve, pay or override a rule, the agent decides from the plan rules in the tool results. | when the user makes such a request | Paying because the user says they are a supervisor |
| `next_step` | The answer gives a concrete action that the member or provider could take to change the outcome, such as filing an appeal. | when the payout is $0 | "Denied, no authorization on file" and nothing more |

<details>
<summary>Why separate yes/no criteria give more useful scores than a 1 to 5 scale</summary>

A single 1 to 5 rating mixes several properties into one number. An answer with a correct payout and no next step and an answer with a wrong payout and a helpful next step can both receive a 3, and the score gives no way to tell which problem each answer has. Each itemized criterion answers one question with a clear meaning, so a FAIL points at the property that went wrong, and a person can label the same question for the same trace to check the judge.

The first two criteria are easy to confuse, because both concern whether the answer can be trusted. An answer is self-consistent and ungrounded when it tells a coherent story built on facts that no tool returned. It is grounded and inconsistent when every figure matches the tool results but the decision label contradicts the payout, for example an `approve` label on a $0 payout.

The two extra verdicts keep questions that do not fit a trace out of the score. If PASS and FAIL were the only allowed answers, the judge would have to return one of them for questions that do not fit the trace, such as `next_step` on an approved claim, and those verdicts would count toward the pass rate.
</details>

## The judge prompt

The prompt contains the full transcript, including every tool call and tool result, followed by the rubric questions. The system prompt tells the judge to treat the tool results as the source of truth and to write its reasoning before each verdict, so that each verdict is generated after, and conditioned on, the reasoning that supports it.

In [ ]:
example = traces["v1__TC-20__t1"]
print(build_prompt(example))

The judge request requires Nova 2 Lite to call a tool named `record_verdicts`, whose input holds a `reasoning` and a `verdict` for each criterion, so every response is a JSON object with the same fields. The code reads the verdicts directly from that tool input.

<details>
<summary>How the verdict tool's JSON schema works</summary>

In the Bedrock Converse API a tool is a function definition that the model can choose to call, and its input schema is a JSON Schema that describes the arguments the model must supply. The input schema of `record_verdicts` has one property for each rubric criterion, and each property is an object with a string `reasoning` and a string `verdict`. The `verdict` field has an `enum` that allows only `PASS`, `FAIL`, `UNKNOWN` and `NOT_APPLICABLE`, and both fields are listed as required. The `reasoning` field comes first in the schema, which matches the instruction to reason before giving the verdict. Setting `toolChoice` to this tool requires Nova 2 Lite to call it instead of replying in prose, so the code reads the verdicts from the tool input without parsing free text. The Going further section prints the schema entry for `next_step`.
</details>

## Judging one trace with Amazon Nova 2 Lite

The cell below sends the baseline TC-20 transcript to Nova 2 Lite and prints each verdict with its reasoning. A model picks each output token with some randomness, and the setting that controls how much is called the temperature. The judge runs at a temperature of 0, which asks for the most likely token at every step.

In [ ]:
result = judge_trace(example)
display(pd.DataFrame({"verdict": result["verdicts"], "reasoning": result["reasoning"]}))

## Judging 12 traces in parallel

Each judge call is independent, so `judge_many` sends up to eight calls at once from separate threads. The cell below judges the first trial of six cases that should be denied, for both `v1` and `v2-nova`.

In [ ]:
denial_cases = ["TC-02", "TC-10", "TC-17", "TC-19", "TC-20", "TC-27"]
sample = [traces[f"{v}__{cid}__t1"] for v in ["v1", "v2-nova"] for cid in denial_cases]

started = time.perf_counter()
live = judge_many(sample)
print(f"judged {len(live)} traces in {time.perf_counter() - started:.1f} s")

pd.DataFrame([{"trial": r["trial_id"], **r["verdicts"]} for r in live])

<div style="background-color: #d4edda; border-left: 4px solid #28a745; padding: 10px; margin: 10px 0;">
Judging all 405 traces takes about two minutes with eight workers, so we ran the same judge on every trace before the session so that everyone works from the same numbers. The next cell loads those results from <code>data/judge_verdicts/original_rubric.jsonl</code> and lists any of your verdicts that differ from them.
</div>

In [ ]:
stored = {r["trial_id"]: r for r in load_jsonl(JUDGE_RESULTS)}
changes = [{"trial": r["trial_id"], "criterion": k, "stored": stored[r["trial_id"]]["verdicts"][k], "your run": v}
           for r in live for k, v in r["verdicts"].items() if v != stored[r["trial_id"]]["verdicts"][k]]
total = len(live) * len(RUBRIC)
print(f"{len(stored)} stored verdict sets. {total - len(changes)} of your {total} verdicts match the stored run.")
pd.DataFrame(changes) if changes else print("Every verdict matches.")

A temperature of 0 makes the judge more repeatable without making it fully deterministic, so a few of your 48 verdicts can differ from the stored run. The differences are usually on `next_step` or `follows_plan_rules`, where the judge switches between `NOT_APPLICABLE` and PASS or FAIL.

<details>
<summary>Temperature and repeatability</summary>

Two requests with the same input and a temperature of 0 can still produce different tokens, and when one early token in the reasoning differs, the rest of the reasoning and the verdict can follow a different path. Before the session we ran the judge a second time over all 405 traces and compared the two runs. Every verdict on `grounded` and `self_consistent` was identical, and the verdicts that changed were on `next_step` and `follows_plan_rules`, where nearly every change was a switch between `NOT_APPLICABLE` and PASS or FAIL. When the same input gets different verdicts on different runs, the rubric line usually supports two readings, and Lab 3 rewrites the applicability rule for `next_step` and measures how closely the revised judge agrees with a human reviewer.

The ClaimClerk versions that use Claude Sonnet 5.5 and Claude Haiku 5.5 run without a temperature setting, because Claude 5.x models on Amazon Bedrock reject the temperature parameter, so their repeated trials of one case can differ. The `v2-nova` agent and the Nova 2 Lite judge both run at a temperature of 0.
</details>

## Judge pass rates by agent version

The chart shows the **pass rate among applicable verdicts** for each version and criterion, which is the share of PASS verdicts among the traces where the criterion applied. `NOT_APPLICABLE` verdicts are left out of both the numerator and the denominator, so a criterion that does not apply to a trace does not change the score, while an `UNKNOWN` verdict counts in the denominator as a verdict that did not pass. The table under the chart gives the number of applicable verdicts for each version and criterion. With $A_{v,c}$ the set of traces of version $v$ where the judge returned a verdict other than `NOT_APPLICABLE` on criterion $c$, the pass rate is

$$\text{pass rate}(v, c) = \frac{\left|\{\, t \in A_{v,c} : \text{verdict}_c(t) = \text{PASS} \,\}\right|}{|A_{v,c}|}.$$

In [ ]:
rates, counts = pass_rates(list(stored.values()))
rate_heatmap(rates, "LLM judge pass rates by agent version",
             subtitle="Share of applicable verdicts that are PASS (Amazon Nova 2 Lite judge, 81 trials per version)")
plt.show()
display(counts.rename(columns=lambda c: f"{c} (n)"))

On `grounded`, `self_consistent` and `follows_plan_rules` the judge returned PASS on every applicable verdict for every version. These 1.00 rates include `v2-tools`, which never calls `check_coverage` and pays 6 of the 11 claims that should be denied, so a rate of 1.00 shows only that the judge reported no failure.

On `next_step` the judge passed 4 of the 22 applicable `v2-nova` verdicts, a rate of 0.18, while the Claude versions `v1`, `v2-prompt` and `v2-model` have rates between 0.54 and 0.63. The judge passed every Claude trial on TC-19, TC-20 and TC-27, and fewer than half of the Claude trials on the six cases with a missing authorization. The three Claude rates are close and each covers a different number of applicable traces, so they do not rank those versions.

<details>
<summary>Why the applicable counts differ between versions</summary>

The rubric says `next_step` applies to every trace with a $0 payout. `v1`, `v2-prompt`, `v2-model` and `v2-nova` each return a $0 payout on 33 trials, which are the three trials of each of the 11 cases that should pay nothing. Nine of those cases are denials that the member or provider can act on, and in the other two (TC-25 and TC-26) the remaining deductible is larger than the bill. The judge treated 19 of the 33 trials as applicable for `v1`, 25 for `v2-prompt`, 26 for `v2-model` and 21 for `v2-nova`. On the six deductible trials of each version it returned a verdict on three for `v2-prompt` and `v2-model` and on none for `v1` and `v2-nova`.

The judge also returned a verdict on some TC-24 trials, which approve a payout of $144 for a specialist visit and deny an MRI in the same claim, so the rule calls for `NOT_APPLICABLE`. It did this on two trials for `v1`, three for `v2-prompt` and one each for `v2-model` and `v2-nova`, which is why the counts in the table are 21, 28, 27 and 22. Some of the `NOT_APPLICABLE` verdicts on $0 answers come with reasoning that says the answer gives no next step, which is the description of a FAIL. The reasoning on `v1__TC-10__t1` is one example, and Lab 3 traces these verdicts to the wording of the applicability rule.

`v2-tools` returns a $0 payout on 15 trials, the three trials of each of TC-19, TC-20, TC-25, TC-26 and TC-27. The judge treated 14 of them as applicable and passed 10, so the `v2-tools` rate of 0.71 comes mostly from the unknown-member and social-engineering cases, where the judge also passed every Claude trial. On `follows_plan_rules` the rule covers the six trials of TC-20 and TC-27, where the user asks for approval. The judge returned a verdict on one more trial for `v2-nova` (TC-29) and two more for `v2-tools` (TC-05 and TC-11), although none of those three prompts contains a request.
</details>

## Conclusions and takeaways

1. An LLM judge can score qualities that have no exact answer to compare against. On TC-20 it passed every `v1` trial and failed every `v2-nova` trial on `next_step`, although Lab 1 gives both versions a composite score of 1.00 on that case.
2. Write the rubric as separate yes/no criteria with applicability rules, and have the judge record its reasoning and verdicts through a tool call that the request requires.
3. Treat judge scores as unverified until you compare them with human labels, starting with criteria that score 1.00 even for a version that Lab 1 shows is broken.

## Next steps

Continue to [Lab 3: Judge calibration and release gate](Lab3_Judge_Calibration.ipynb).

## Going further (optional, not needed in the session)

This section contains optional code cells, and Lab 3 does not depend on any of them. The cells use variables from the core lab, including the stored judge results, so run the core cells first.

### The schema entry for `next_step`

The cell below prints the part of the `record_verdicts` input schema that describes one criterion. The other three criteria have identical entries.

In [ ]:
print(json.dumps(verdict_tool(RUBRIC)["toolSpec"]["inputSchema"]["json"]["properties"]["next_step"], indent=2))

### A `v2-tools` answer that passes `grounded`

The `v2-tools` agent has no access to `check_coverage`, and on TC-10 it approves a CT scan that requires a prior authorization that is not on file. The trace below is its first trial. The answer reports the $1,600 payout that `compute_payout` returned and then says that the authorization status could not be checked because the coverage tool was unavailable. Every figure in the answer appears in a tool result or follows from one, so a PASS on `grounded` agrees with the rubric's definition, even though the decision is wrong and Lab 1 gives all three TC-10 trials of `v2-tools` a composite score of 0. None of the four criteria asks whether the agent called every tool that the decision requires, so the judge's scores do not reflect this failure, which the code-based evaluators in Lab 1 report.

In [ ]:
show_trace(traces["v2-tools__TC-10__t1"], cases["TC-10"])
print("grounded:", stored["v2-tools__TC-10__t1"]["verdicts"]["grounded"])
print(stored["v2-tools__TC-10__t1"]["reasoning"]["grounded"])

### Judge cost and latency

A token is the unit of text that a model reads and writes, usually a short word or part of a word. Every judge call sends the full transcript as input and receives a short set of verdicts as output, so each call uses several times more input tokens than output tokens. Bedrock charges separate prices for input and output tokens, so with $N_{\text{in}}$ and $N_{\text{out}}$ the total token counts over the stored run of 405 traces, and $p_{\text{in}}$ and $p_{\text{out}}$ the prices per token for Nova 2 Lite from the current [Amazon Bedrock pricing page](https://aws.amazon.com/bedrock/pricing/), the cost of one full evaluation run is

$$\text{cost} = N_{\text{in}} \, p_{\text{in}} + N_{\text{out}} \, p_{\text{out}}.$$

Latency is the time from sending a request to receiving the complete response. The cell reports the median latency per call and the 95th percentile, which is the latency that 95 percent of calls complete within.

In [ ]:
rows = list(stored.values())
tokens_in = sum(r["usage"]["input_tokens"] for r in rows)
tokens_out = sum(r["usage"]["output_tokens"] for r in rows)
latency = pd.Series([r["latency_s"] for r in rows])
print(f"{len(rows)} judge calls: {tokens_in:,} input tokens, {tokens_out:,} output tokens")
print(f"latency per call: median {latency.median():.1f} s, 95th percentile {latency.quantile(0.95):.1f} s")